<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/03_guard_rails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalação das bibliotecas necessárias
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama langchain-community langchain-chroma langchain-core

# 2. Inicialização do servidor Ollama em segundo plano
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 3. Download do modelo
!ollama pull qwen2.5:3b

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,943 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,543 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [

In [3]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate

# 1. Configuração do modelo com temperatura 0 (zero criatividade/alucinação)
llm = Ollama(model="qwen2.5:3b", temperature=0)

# 2. Prompt com Guard-Rail reforçado
template_guard_rail = """
[INSTRUÇÃO CRÍTICA DE SEGURANÇA]
Você é um sistema restrito. Responda APENAS usando o contexto fornecido.
Se a informação NÃO estiver explicitamente escrita no contexto, sua ÚNICA resposta deve ser:
"Desculpe, não encontrei essa informação no material fornecido."

NÃO utilize nenhum conhecimento prévio ou externo sob nenhuma circunstância.

Contexto:
{contexto}

Pergunta:
{pergunta}

Resposta:
"""

prompt = ChatPromptTemplate.from_template(template_guard_rail)
chain = prompt | llm

# 3. Teste de Validação
contexto_teste = "O projeto EduAcessível utiliza Inteligência Artificial para adaptar materiais pedagógicos para estudantes com necessidades especiais."

print("--- Teste 1: Pergunta presente no contexto ---")
pergunta_valida = "Qual é o objetivo do projeto EduAcessível?"
resposta_1 = chain.invoke({"contexto": contexto_teste, "pergunta": pergunta_valida})
print(resposta_1)

print("\n--- Teste 2: Pergunta ausente no contexto (Disparo do Guard-Rail) ---")
pergunta_invalida = "Qual é a capital da França?"
resposta_2 = chain.invoke({"contexto": contexto_teste, "pergunta": pergunta_invalida})
print(resposta_2)

--- Teste 1: Pergunta presente no contexto ---
O objetivo do projeto EduAcessível é utilizar Inteligência Artificial para adaptar materiais pedagógicos para estudantes com necessidades especiais.

--- Teste 2: Pergunta ausente no contexto (Disparo do Guard-Rail) ---
Desculpa, não encontrei essa informação no material fornecido.
